# 📊 Análisis de Datos con Google BigQuery
### Plataforma DataHub EIA — datahub.eia.edu.co

---

Este notebook muestra cómo consultar y analizar grandes volúmenes de datos directamente desde JupyterHub usando **Google BigQuery**, sin necesidad de descargar archivos ni configurar credenciales manualmente.

**Lo que aprenderás:**
- Conectarte a BigQuery desde Python
- Consultar datasets públicos con SQL
- Traer los resultados como DataFrame de pandas
- Visualizar los datos con gráficas

**Datasets que usaremos:** `bigquery-public-data` — datos públicos gratuitos que Google pone a disposición de todos los usuarios de GCP.

> ✅ **No necesitas configurar nada.** La autenticación con GCP está integrada en la plataforma DataHub EIA.

---
## 1. Instalación y conexión

Si los paquetes no están instalados en tu entorno, ejecuta la siguiente celda una vez:

In [ ]:
# Solo si es necesario — en el perfil de Ciencia de Datos EIA ya vienen instalados
# !pip install --quiet google-cloud-bigquery google-cloud-bigquery-storage pandas-gbq pyarrow db-dtypes

In [ ]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ─── Conexión a BigQuery ───────────────────────────────────────────────────────
# El proyecto de facturación es el de DataHub EIA.
# Las consultas sobre bigquery-public-data son gratuitas (hasta 1 TB/mes).
PROJECT_ID = "jupyterhub-eia"   # ← proyecto GCP de DataHub EIA

client = bigquery.Client(project=PROJECT_ID)
print(f"✅ Conexión exitosa al proyecto: {PROJECT_ID}")
print(f"   Versión de la librería: {bigquery.__version__}")

---
## 2. Primera consulta: explorar un dataset público

Vamos a usar el dataset de **calidad del aire** de la EPA (Agencia de Protección Ambiental de EE.UU.), que contiene mediciones históricas de contaminantes en ciudades de todo el mundo.

En BigQuery, una tabla se referencia como: `proyecto.dataset.tabla`

In [ ]:
# ─── Ver qué tablas tiene el dataset de calidad del aire ──────────────────────
query_tablas = """
SELECT table_name, table_type, creation_time
FROM `bigquery-public-data.epa_historical_air_quality.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
LIMIT 10
"""

df_tablas = client.query(query_tablas).to_dataframe(create_bqstorage_client=False)
df_tablas

---
## 3. Análisis de PM2.5 — Partículas finas en el aire

El **PM2.5** (material particulado fino) es uno de los contaminantes más peligrosos para la salud. Vamos a analizar su evolución histórica en ciudades de Estados Unidos.

### 3.1 Promedio anual de PM2.5 por estado

In [ ]:
query_pm25_anual = """
SELECT
    EXTRACT(YEAR FROM date_local) AS anio,
    state_name                    AS estado,
    ROUND(AVG(arithmetic_mean), 3) AS pm25_promedio
FROM
    `bigquery-public-data.epa_historical_air_quality.pm25_frm_daily_summary`
WHERE
    state_name IN ('California', 'Texas', 'New York', 'Florida', 'Illinois')
    AND EXTRACT(YEAR FROM date_local) BETWEEN 2010 AND 2022
    AND arithmetic_mean IS NOT NULL
    AND arithmetic_mean >= 0
GROUP BY
    anio, estado
ORDER BY
    anio, estado
"""

print("⏳ Consultando BigQuery...")
df_pm25 = client.query(query_pm25_anual).to_dataframe(create_bqstorage_client=False)
print(f"✅ {len(df_pm25):,} filas obtenidas")
df_pm25.head(10)

In [ ]:
# ─── Visualización ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

colores = {
    'California': '#E63946',
    'Texas':      '#457B9D',
    'New York':   '#2A9D8F',
    'Florida':    '#E9C46A',
    'Illinois':   '#9B5DE5',
}

for estado, grupo in df_pm25.groupby('estado'):
    ax.plot(grupo['anio'], grupo['pm25_promedio'],
            marker='o', linewidth=2, markersize=5,
            label=estado, color=colores.get(estado, 'gray'))

# Línea de referencia OMS: 5 µg/m³ (guía 2021)
ax.axhline(y=5, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='Límite OMS (5 µg/m³)')

ax.set_title('Evolución del PM2.5 promedio anual por estado (2010–2022)', fontsize=14, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('PM2.5 promedio (µg/m³)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('pm25_por_estado.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Gráfica guardada como pm25_por_estado.png")

---
### 3.2 Peores ciudades en PM2.5 (promedio 2018–2022)

In [ ]:
query_ciudades = """
SELECT
    city_name                      AS ciudad,
    state_name                     AS estado,
    ROUND(AVG(arithmetic_mean), 2) AS pm25_promedio,
    COUNT(*)                       AS dias_medidos
FROM
    `bigquery-public-data.epa_historical_air_quality.pm25_frm_daily_summary`
WHERE
    EXTRACT(YEAR FROM date_local) BETWEEN 2018 AND 2022
    AND arithmetic_mean > 0
    AND city_name != 'Not in a city'
GROUP BY
    ciudad, estado
HAVING
    dias_medidos > 200   -- Solo ciudades con mediciones consistentes
ORDER BY
    pm25_promedio DESC
LIMIT 15
"""

df_ciudades = client.query(query_ciudades).to_dataframe(create_bqstorage_client=False)
df_ciudades

In [ ]:
# ─── Gráfica de barras horizontal ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

colores_barras = ['#E63946' if v > 10 else '#457B9D' for v in df_ciudades['pm25_promedio']]

bars = ax.barh(
    df_ciudades['ciudad'] + ' (' + df_ciudades['estado'].str[:2] + ')',
    df_ciudades['pm25_promedio'],
    color=colores_barras, edgecolor='white', height=0.7
)

ax.axvline(x=5,  color='orange', linestyle='--', linewidth=1.2, alpha=0.8, label='Límite OMS 2021 (5 µg/m³)')
ax.axvline(x=12, color='red',    linestyle='--', linewidth=1.2, alpha=0.8, label='Límite EPA (12 µg/m³)')

# Etiquetas de valor
for bar, val in zip(bars, df_ciudades['pm25_promedio']):
    ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

ax.set_title('Top 15 ciudades con mayor PM2.5 promedio (2018–2022)', fontsize=13, fontweight='bold')
ax.set_xlabel('PM2.5 promedio (µg/m³)')
ax.invert_yaxis()
ax.legend(fontsize=9)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('pm25_ciudades.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Exportar resultados

Los resultados de BigQuery son DataFrames normales de pandas. Puedes exportarlos como CSV, Excel, o guardarlos de vuelta en GCS.

In [ ]:
# ─── Guardar como CSV en tu almacenamiento persistente ────────────────────────
df_pm25.to_csv('pm25_anual_por_estado.csv', index=False)
df_ciudades.to_csv('pm25_ciudades_ranking.csv', index=False)

print("✅ Archivos guardados en tu espacio personal:")
print("   • pm25_anual_por_estado.csv")
print("   • pm25_ciudades_ranking.csv")
print("   • pm25_por_estado.png")
print("   • pm25_ciudades.png")
print()
print("📌 Estos archivos persisten entre sesiones en tu almacenamiento de 3 GB.")

---
## 5. Conceptos clave de BigQuery

### ¿Cómo se cobra BigQuery?

BigQuery cobra por **datos procesados** (escaneados) por cada consulta, no por tiempo de ejecución.

| Tier | Límite | Costo |
|------|--------|-------|
| Gratuito | 1 TB procesado / mes | $0 |
| On-demand | > 1 TB | ~$5 por TB adicional |

Las consultas sobre `bigquery-public-data` cuentan contra tu cuota gratuita. Para este curso, **no hay costo**.

### Buenas prácticas para no gastar créditos innecesariamente

```sql
-- ✅ BIEN: Selecciona solo las columnas que necesitas
SELECT date_local, arithmetic_mean FROM tabla WHERE ...

-- ❌ MAL: SELECT * escanea todas las columnas (más datos = más costo)
SELECT * FROM tabla

-- ✅ BIEN: Filtra por fecha/partición cuando sea posible
WHERE EXTRACT(YEAR FROM date_local) = 2022

-- ✅ BIEN: Usa LIMIT para exploración inicial
SELECT * FROM tabla LIMIT 100
```

### Datasets públicos útiles para clases

| Dataset | Descripción | Referencia BQ |
|---------|-------------|---------------|
| Calidad del aire EPA | Mediciones históricas USA | `bigquery-public-data.epa_historical_air_quality` |
| Nombres en USA | Frecuencia de nombres 1910–2013 | `bigquery-public-data.usa_names` |
| COVID-19 | Casos globales por país | `bigquery-public-data.covid19_open_data` |
| Stack Overflow | Posts y respuestas 2008–2023 | `bigquery-public-data.stackoverflow` |
| Wikipedia | Tráfico por artículo | `bigquery-public-data.wikipedia` |
| NOAA Climate | Datos climáticos globales | `bigquery-public-data.noaa_gsod` |
| NYC Taxi | Viajes en taxi Nueva York | `bigquery-public-data.new_york_taxi_trips` |

---
## 6. Ejercicio propuesto para la clase

Usando el dataset de **NYC Taxi Trips** (`bigquery-public-data.new_york_taxi_trips`), responde:

1. ¿Cuál es la distancia promedio de un viaje en taxi en Nueva York?
2. ¿En qué hora del día se registran más viajes?
3. ¿Cuál es la propina promedio según el método de pago?

**Pista para empezar:**

In [ ]:
# ─── Explorar la tabla de taxis de NYC ────────────────────────────────────────
query_taxi_preview = """
SELECT *
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022`
LIMIT 5
"""

df_taxi = client.query(query_taxi_preview).to_dataframe(create_bqstorage_client=False)
print("Columnas disponibles:")
print(df_taxi.columns.tolist())
df_taxi.head()

In [ ]:
# ─── Tu solución aquí ─────────────────────────────────────────────────────────

# Pregunta 1: distancia promedio
query_distancia = """
-- Escribe tu consulta aquí
"""

# df_distancia = client.query(query_distancia).to_dataframe()
# df_distancia

---

## 📁 Acceso al bucket compartido

DataHub EIA tiene un bucket compartido montado en `/srv/shared/`. Ahí puedes encontrar datasets y notebooks precargados por el equipo docente.


In [ ]:
import os

# ─── Ver contenido del bucket compartido ──────────────────────────────────────
shared_path = '/srv/shared'

if os.path.exists(shared_path):
    print(f"📂 Contenido de {shared_path}:\n")
    for root, dirs, files in os.walk(shared_path):
        level = root.replace(shared_path, '').count(os.sep)
        indent = '  ' * level
        print(f'{indent}📁 {os.path.basename(root)}/')
        for f in files:
            print(f'{indent}  📄 {f}')
else:
    print("⚠️ El bucket compartido no está montado en este entorno.")
    print("   Verifica con el administrador que el perfil tiene acceso a /srv/shared.")

---

## ℹ️ Información técnica de la sesión

In [ ]:
import sys, platform

print("=" * 45)
print("  DataHub EIA — Información de sesión")
print("=" * 45)
print(f"  Python:         {sys.version.split()[0]}")
print(f"  Sistema:        {platform.system()} {platform.release()}")
print(f"  Proyecto GCP:   {PROJECT_ID}")
print(f"  BigQuery SDK:   {bigquery.__version__}")
print(f"  Pandas:         {pd.__version__}")
print(f"  Almacenamiento: /home/jovyan (3 GB persistente)")
print(f"  Shared bucket:  /srv/shared (solo lectura)")
print("=" * 45)
print("  Contacto: jaime.sanchez@eia.edu.co")
print("=" * 45)